# EDA: Central Required Features Raw

Notebook นี้ใช้สำรวจไฟล์ `data/raw/central/central_required_features_raw.csv` ซึ่งเป็น data กลางดิบตาม feature ที่ต้องการใช้จริง:

- Area size
- Soil type
- pH
- NPK
- Soil humidity
- Temp
- Rainfall

ขอบเขตคือ EDA เบื้องต้นเท่านั้น ยังไม่ clean, impute, convert unit หรือ merge เติมค่าข้าม source


In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "central" / "central_required_features_raw.csv"
SUMMARY_PATH = PROJECT_ROOT / "data" / "raw" / "central" / "central_required_features_summary.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

df = pd.read_csv(DATA_PATH, low_memory=False)
summary_by_source = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.exists() else None

print("Data path:", DATA_PATH.relative_to(PROJECT_ROOT))
print("Shape:", df.shape)


Data path: data/raw/central/central_required_features_raw.csv
Shape: (53963, 34)


## 1. File overview

ดูขนาดไฟล์ จำนวนแถว จำนวนคอลัมน์ และตัวอย่างข้อมูลชุดแรก เพื่อเช็กว่าอ่านไฟล์ถูกต้อง


In [13]:
file_overview = pd.DataFrame({
    "metric": ["file", "size_mb", "rows", "columns", "memory_mb"],
    "value": [
        DATA_PATH.relative_to(PROJECT_ROOT).as_posix(),
        round(DATA_PATH.stat().st_size / (1024 ** 2), 4),
        len(df),
        len(df.columns),
        round(df.memory_usage(deep=True).sum() / (1024 ** 2), 4),
    ],
})
display(file_overview)
display(df.head(10))


,metric,value
0,file,data/raw/central/central_required_features_raw...
1,size_mb,17.2440
2,rows,53963
3,columns,34
4,memory_mb,59.4522


,central_feature_id,source_file,source_path,source_row_id,source_record_hash,source_note,label,area_size,area_size_unit,soil_type,ph,ph_unit,n,n_unit,p,p_unit,k,k_unit,soil_humidity,soil_humidity_unit,temp,temp_unit,rainfall,rainfall_unit,has_area_size,has_soil_type,has_ph,has_npk,has_soil_humidity,has_temp,has_rainfall,required_feature_count,is_required_feature_complete,missing_required_features
0,CFEAT-000001,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,1,68df9bd8ee14b97d2b71b28cd516081570cc486901a5ce...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.8100,pH,0.2300,%,5.4010,ppm,738.2310,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
1,CFEAT-000002,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,2,48c29067b5ede0746c3454f38cc088dbd55e5961c2f57d...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.4300,pH,0.2300,%,10.4780,ppm,606.3820,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
2,CFEAT-000003,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,3,40431f5c6db94de505f9ce9a38e1533bb9a1f38b5b157d...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.4100,pH,0.2300,%,6.8470,ppm,386.5800,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
3,CFEAT-000004,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,4,3cb32c6b9d66bc99da0ae4bf8bfdb2ecf98f246b7d0f99...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.6500,pH,0.2300,%,3.4180,ppm,207.0860,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
4,CFEAT-000005,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,5,7080b59f17574b87ef1973e7aacf11343b697a8e06691c...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.2700,pH,0.2300,%,39.2820,ppm,317.3570,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
5,CFEAT-000006,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,6,d7d6783bfcef75184f115c9f77e7a65e6f8c83328dcfa8...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.0600,pH,0.2300,%,8.4800,ppm,148.4370,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
6,CFEAT-000007,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,7,de68d07d5998c3152948a87859abb5b619d444754e846c...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.5700,pH,0.2300,%,2.1470,ppm,303.5280,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
7,CFEAT-000008,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,8,6ccf691c48b79972559234b25848d2afc83937a0269417...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.2000,pH,0.2300,%,30.3000,ppm,380.5990,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
8,CFEAT-000009,Crop Recommendation using Soil Properties and ...,data/raw/pre-combine/Crop Recommendation using...,9,eeb6bba00bb1f4900ccee1cb9951ea0f2d1240a0d647ac...,N is percent; P and K are ppm. Weather feature...,barley,NaN,NaN,NaN,5.3400,pH,0.2300,%,6.3820,ppm,401.0630,ppm,0.7300,fraction_0_1,NaN,NaN,NaN,NaN,False,False,True,True,True,False,False,3,False,area_size;soil_type;temp;rainfall
9,CFEAT-000010,Crop Recommendatio

## 2. Schema และชนิดข้อมูล

ตรวจ column, dtype, missing count, missing percent และจำนวนค่า unique ของแต่ละคอลัมน์


In [14]:
schema_profile = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique_count": df.nunique(dropna=True).values,
})
display(schema_profile)


,column,dtype,missing_count,missing_pct,unique_count
0,central_feature_id,str,0,0.0000,53963
1,source_file,str,0,0.0000,8
2,source_path,str,0,0.0000,8
3,source_row_id,int64,0,0.0000,19689
4,source_record_hash,str,0,0.0000,49915
5,source_note,str,42896,79.4900,3
6,label,str,0,0.0000,159
7,area_size,float64,34274,63.5100,13644
8,area_size_unit,str,34274,63.5100,1
9,soil_type,str,30963,57.3800,11


## 3. Missing values ของ feature หลัก

ดูว่า feature ตามโจทย์ขาดมากน้อยแค่ไหน โดยแยกตาม column หลักและตาม source file


In [15]:
required_features = ["area_size", "soil_type", "ph", "n", "p", "k", "soil_humidity", "temp", "rainfall"]

missing_required = pd.DataFrame({
    "feature": required_features,
    "non_null_count": [int(df[col].notna().sum()) for col in required_features],
    "missing_count": [int(df[col].isna().sum()) for col in required_features],
    "missing_pct": [(df[col].isna().mean() * 100).round(2) for col in required_features],
})
display(missing_required)

missing_by_source = (
    df.groupby("source_file")[required_features]
    .apply(lambda g: g.isna().mean().mul(100).round(2))
    .reset_index()
)
display(missing_by_source)


,feature,non_null_count,missing_count,missing_pct
0,area_size,19689,34274,63.5100
1,soil_type,23000,30963,57.3800
2,ph,45963,8000,14.8200
3,n,53963,0,0.0000
4,p,53963,0,0.0000
5,k,53963,0,0.0000
6,soil_humidity,21867,32096,59.4800
7,temp,50096,3867,7.1700
8,rainfall,42096,11867,21.9900


,source_file,area_size,soil_type,ph,n,p,k,soil_humidity,temp,rainfall
0,Crop Recommendation using Soil Properties and ...,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,100.0000,100.0000
1,Crop and fertilizer dataset.csv,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,100.0000,0.0000,0.0000
2,Crop_Recommendation.csv,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,100.0000,0.0000,0.0000
3,crop-yield.csv,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,crop_data.csv,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,100.0000,0.0000,0.0000
5,crop_yield.csv,0.0000,100.0000,0.0000,0.0000,0.0000,0.0000,100.0000,0.0000,0.0000
6,data_core.csv,100.0000,0.0000,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,100.0000
7,original_dataset.csv,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,100.0000,0.0000,0.0000


## 4. Completeness ตาม feature ที่ต้องใช้

ไฟล์นี้มี flag `has_*` และ `missing_required_features` เพื่อบอกว่าแต่ละ row มี feature ครบแค่ไหน โดยยังไม่เติมค่าข้าม source


In [16]:
feature_flags = [
    "has_area_size", "has_soil_type", "has_ph", "has_npk",
    "has_soil_humidity", "has_temp", "has_rainfall",
]

completeness_overview = pd.DataFrame({
    "metric": ["complete_rows", "incomplete_rows", "complete_pct", "avg_required_feature_count"],
    "value": [
        int(df["is_required_feature_complete"].sum()),
        int((~df["is_required_feature_complete"]).sum()),
        round(df["is_required_feature_complete"].mean() * 100, 4),
        round(df["required_feature_count"].mean(), 4),
    ],
})
display(completeness_overview)

display(df["required_feature_count"].value_counts().sort_index().to_frame("rows"))
display(df["missing_required_features"].value_counts().to_frame("rows"))

if summary_by_source is not None:
    display(summary_by_source)


,metric,value
0,complete_rows,0.0000
1,incomplete_rows,"53,963.0000"
2,complete_pct,0.0000
3,avg_required_feature_count,4.7565


,rows
required_feature_count,
3,3867
4,15407
5,24689
6,10000


,rows
missing_required_features,
soil_type;soil_humidity,19689
area_size,10000
area_size;ph;rainfall,8000
area_size;soil_type;soil_humidity,7407
area_size;soil_humidity,5000
area_size;soil_type;temp;rainfall,3867


,source_file,rows,complete_required_rows,avg_required_feature_count,has_area_size_rows,has_soil_type_rows,has_ph_rows,has_npk_rows,has_soil_humidity_rows,has_temp_rows,has_rainfall_rows
0,Crop Recommendation using Soil Properties and ...,3867,0,3.0000,0,0,3867,3867,3867,0,0
1,Crop and fertilizer dataset.csv,4513,0,4.0000,0,0,4513,4513,0,4513,4513
2,Crop_Recommendation.csv,2200,0,4.0000,0,0,2200,2200,0,2200,2200
3,crop-yield.csv,10000,0,6.0000,0,10000,10000,10000,10000,10000,10000
4,crop_data.csv,694,0,4.0000,0,0,694,694,0,694,694
5,crop_yield.csv,19689,0,5.0000,19689,0,19689,19689,0,19689,19689
6,data_core.csv,8000,0,4.0000,0,8000,0,8000,8000,8000,0
7,original_dataset.csv,5000,0,5.0000,0,5000,5000,5000,0,5000,5000


## 5. Source distribution

ดูจำนวนแถวจากแต่ละไฟล์ต้นทาง และ label/crop ที่พบมากที่สุดในแต่ละ source


In [17]:
source_counts = df["source_file"].value_counts().rename_axis("source_file").reset_index(name="rows")
source_counts["pct"] = (source_counts["rows"] / len(df) * 100).round(2)
display(source_counts)

label_by_source = (
    df.groupby(["source_file", "label"])
    .size()
    .reset_index(name="rows")
    .sort_values(["source_file", "rows"], ascending=[True, False])
)
display(label_by_source.groupby("source_file").head(10).reset_index(drop=True))


,source_file,rows,pct
0,crop_yield.csv,19689,36.4900
1,crop-yield.csv,10000,18.5300
2,data_core.csv,8000,14.8200
3,original_dataset.csv,5000,9.2700
4,Crop and fertilizer dataset.csv,4513,8.3600
5,Crop Recommendation using Soil Properties and ...,3867,7.1700
6,Crop_Recommendation.csv,2200,4.0800
7,crop_data.csv,694,1.2900


,source_file,label,rows
0,Crop Recommendation using Soil Properties and ...,teff,1260
1,Crop Recommendation using Soil Properties and ...,maize,732
2,Crop Recommendation using Soil Properties and ...,wheat,715
3,Crop Recommendation using Soil Properties and ...,barley,503
4,Crop Recommendation using Soil Properties and ...,bean,253
5,Crop Recommendation using Soil Properties and ...,pea,94
6,Crop Recommendation using Soil Properties and ...,sorghum,72
7,Crop Recommendation using Soil Properties and ...,dagussa,71
8,Crop Recommendation using Soil Properties and ...,niger_seed,64
9,Crop Recommendation using Soil Properties and ...,potato,48


## 6. Duplicate และ hash check

เช็ก ID, source row key และ hash ของ record ต้นฉบับ เพื่อดูว่ามี row ที่ซ้ำจากต้นทางหรือไม่


In [18]:
duplicate_checks = pd.DataFrame({
    "metric": [
        "central_feature_id_duplicates",
        "source_file_source_row_id_duplicates",
        "source_record_hash_duplicates",
        "exact_duplicate_required_feature_rows",
    ],
    "value": [
        int(df["central_feature_id"].duplicated().sum()),
        int(df.duplicated(["source_file", "source_row_id"]).sum()),
        int(df["source_record_hash"].duplicated().sum()),
        int(df[required_features].duplicated().sum()),
    ],
})
display(duplicate_checks)

dupe_hash = df[df["source_record_hash"].duplicated(keep=False)].sort_values("source_record_hash")
display(dupe_hash[["central_feature_id", "source_file", "source_row_id", "label", "source_record_hash"]].head(20))


,metric,value
0,central_feature_id_duplicates,0
1,source_file_source_row_id_duplicates,0
2,source_record_hash_duplicates,4048
3,exact_duplicate_required_feature_rows,4488


,central_feature_id,source_file,source_row_id,label,source_record_hash
53721,CFEAT-053722,original_dataset.csv,4759,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
52037,CFEAT-052038,original_dataset.csv,3075,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
48970,CFEAT-048971,original_dataset.csv,8,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
52184,CFEAT-052185,original_dataset.csv,3222,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
51026,CFEAT-051027,original_dataset.csv,2064,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
49675,CFEAT-049676,original_dataset.csv,713,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
51098,CFEAT-051099,original_dataset.csv,2136,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
51985,CFEAT-051986,original_dataset.csv,3023,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
51355,CFEAT-051356,original_dataset.csv,2393,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...
52073,CFEAT-052074,original_dataset.csv,3111,maize,0058ec539edecabd75fbe70cdd170bce58bf618b800801...


## 7. Unit consistency

ตรวจหน่วยของ N/P/K, soil humidity, temp และ rainfall แยกตาม source เพื่อกันการตีความผิดก่อน clean/convert


In [19]:
unit_cols = [
    "area_size_unit", "ph_unit", "n_unit", "p_unit", "k_unit",
    "soil_humidity_unit", "temp_unit", "rainfall_unit",
]

unit_by_source = df.groupby("source_file")[unit_cols].agg(
    lambda s: "; ".join(sorted(map(str, s.dropna().unique())))
).reset_index()
display(unit_by_source)

for col in unit_cols:
    display(Markdown(f"**{col}**"))
    display(df[col].value_counts(dropna=False).to_frame("rows"))


,source_file,area_size_unit,ph_unit,n_unit,p_unit,k_unit,soil_humidity_unit,temp_unit,rainfall_unit
0,Crop Recommendation using Soil Properties and ...,,pH,%,ppm,ppm,fraction_0_1,,
1,Crop and fertilizer dataset.csv,,pH,kg/ha,kg/ha,kg/ha,,degC,mm/year
2,Crop_Recommendation.csv,,pH,kg/ha,kg/ha,kg/ha,,degC,unknown_low_scale_not_converted
3,crop-yield.csv,,pH,kg/ha,kg/ha,kg/ha,%,degC,mm/year
4,crop_data.csv,,pH,kg/ha,kg/ha,kg/ha,,degC,mm/year
5,crop_yield.csv,unknown,pH,kg/ha,kg/ha,kg/ha,,degC,mm/year
6,data_core.csv,,,kg/ha,kg/ha,kg/ha,%,degC,
7,original_dataset.csv,,pH,kg/ha_assumed_from_source_family,kg/ha_assumed_from_source_family,kg/ha_assumed_from_source_family,,degC,unknown_low_scale_not_converted


**area_size_unit**

,rows
area_size_unit,
NaN,34274
unknown,19689


**ph_unit**

,rows
ph_unit,
pH,45963
NaN,8000


...

,rows
soil_humidity_unit,
NaN,32096
%,18000
fraction_0_1,3867


**temp_unit**

,rows
temp_unit,
degC,50096
NaN,3867


**rainfall_unit**

,rows
rainfall_unit,
mm/year,34896
NaN,11867
unknown_low_scale_not_converted,7200


## 8. Numeric summary

สถิติพื้นฐานของ feature ตัวเลข โดยแยกดูรวมทั้งไฟล์และดูแยกตาม source file แบบย่อ


In [20]:
numeric_features = ["area_size", "ph", "n", "p", "k", "soil_humidity", "temp", "rainfall"]
display(df[numeric_features].describe().T)

source_numeric_summary = df.groupby("source_file")[numeric_features].agg(["count", "min", "median", "max"])
display(source_numeric_summary)


,count,mean,std,min,25%,50%,75%,max
area_size,"19,689.0000","179,926.5703","732,828.6759",0.5000,"1,390.0000","9,317.0000","75,112.0000","50,808,100.0000"
ph,"45,963.0000",6.5260,0.7894,3.5048,5.9000,6.5000,7.0500,9.9351
n,"53,963.0000",65.9152,43.5579,0.0000,31.0000,68.0000,88.0000,250.0000
p,"53,963.0000",37.3951,24.5892,0.0000,20.0000,37.0000,50.0000,782.0000
k,"53,963.0000",62.8537,97.1635,0.0000,22.0000,36.0000,60.0000,"2,119.0000"
soil_humidity,"21,867.0000",34.4001,20.0009,0.5200,21.8850,36.9500,50.0200,70.0000
temp,"50,096.0000",25.4757,6.1331,6.2600,22.4100,25.8500,28.5200,43.6755
rainfall,"42,096.0000","1,112.2218",762.0228,15.3400,600.0000,"1,000.0000","1,559.7700","5,244.3600"


area_size         \
                                                       count    min   
source_file                                                           
Crop Recommendation using Soil Properties and W...         0    NaN   
Crop and fertilizer dataset.csv                            0    NaN   
Crop_Recommendation.csv                                    0    NaN   
crop-yield.csv                                             0    NaN   
crop_data.csv                                              0    NaN   
crop_yield.csv                                         19689 0.5000   
data_core.csv                                              0    NaN   
original_dataset.csv                                       0    NaN   

                                                                               \
                                                       median             max   
source_file                                                                     
Crop Recommendation using Soil Properties and W...        NaN             NaN   
Crop and fertilizer dataset.csv                           NaN             NaN   
Crop_Recommendation.csv                                   NaN             NaN   
crop-yield.csv                                            NaN             NaN   
crop_data.csv                                             NaN             NaN   
crop_yield.csv                                     9,317.0000 50,808,100.0000   
data_core.csv                                             NaN             NaN   
original_dataset.csv                                      NaN             NaN   

                                                       ph                \
                                                    count    min median   
source_file                                                               
Crop Recommendation using Soil Properties and W...   3867 4.3000 5.7800   
Crop and fertilizer dataset.csv                      4513 5.5000 6.5000   
Crop_Recommendation.csv                              2200 3.5048 6.4250   
crop-yield.csv                                      10000 4.8000 6.5200   
crop_data.csv                                         694 3.8200 6.1600   
crop_yield.csv                                      19689 5.5000 6.6000   
data_core.csv                                           0    NaN    NaN   
original_dataset.csv                                 5000 5.0100 6.5000   

                                                               n          \
                                                      max  count     min   
source_file                                                                
Crop Recommendation using Soil Properties and W... 8.5000   3867  0.0003   
Crop and fertilizer dataset.csv                    8.5000   4513 20.0000   
Crop_Recommendation.csv                            9.9351   2200  0.0000   
crop-yield.csv                                     8.2000  10000 30.0000   
crop_data.csv                                      7.6000    694 10.0000   
crop_yield.csv                                     8.0000  19689 50.0000   
data_core.csv                                         NaN   8000  0.0000   
original_dataset.csv                               8.8700   5000  0.0000   

                                                                          p  \
                                                     median      max  count   
source_file                                                                   
Crop Recommendation using Soil Properties and W...   0.1799   0.6956   3867   
Crop and fertilizer dataset.csv                    105.0000 150.0000   4513   
Crop_Recommendation.csv                             37.0000 140.0000   2200   
crop-yield.csv                                     105.0000 179.0000  10000   
crop_data.csv                                       80.0000 250.0000    694   
crop_yield.csv                                      72.0000 150.0000  19689   
dat

## 9. Categorical summary

ดูค่า categorical หลัก เช่น label, soil type และ note จาก source


In [21]:
categorical_cols = ["label", "soil_type", "source_file", "source_note"]
display(df[categorical_cols].describe(include="object").T)

for col in ["label", "soil_type", "source_note"]:
    display(Markdown(f"**Top values: `{col}`**"))
    display(df[col].value_counts(dropna=False).head(30).to_frame("rows"))


/var/folders/lw/6dtzgyh93g5cx9fllxstd0bc0000gn/T/ipykernel_39948/2861009427.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df[categorical_cols].describe(include="object").T)


,count,unique,top,freq
label,53963,159,maize,5063
soil_type,23000,11,Loamy,4133
source_file,53963,8,crop_yield.csv,19689
source_note,11067,3,Rainfall max is below 300 and unit was not pro...,5000


**Top values: `label`**

,rows
label,
maize,5063
wheat,4547
sugarcane,4021
rice,3780
cotton,3148
potato,2356
barley,1508
teff,1265
tobacco,1081


**Top values: `soil_type`**

,rows
soil_type,
NaN,30963
Loamy,4133
Sandy,4076
Silt,2494
Clay,2467
sandy loam,1836
Clayey,1623
Black,1613
Red,1594


**Top values: `source_note`**

,rows
source_note,
NaN,42896
Rainfall max is below 300 and unit was not provided in the prompt; kept unconverted.,5000
"N is percent; P and K are ppm. Weather features are seasonal, so canonical temperature/rainfall are left blank and original columns are preserved.",3867
Rainfall max is below 300 in this source; kept as raw low-scale rainfall and not converted to yearly millimeters.,2200


## 10. Quick takeaways checklist

รัน cell นี้เพื่อดูประเด็นสั้นๆ ที่ควรใช้ตัดสินใจก่อนทำขั้นตอน clean/merge/enrich ต่อ


In [22]:
takeaways = []
takeaways.append(f"Total rows: {len(df):,}")
takeaways.append(f"Complete required-feature rows: {int(df['is_required_feature_complete'].sum()):,}")
takeaways.append(f"Sources: {df['source_file'].nunique():,}")
takeaways.append(f"Labels/crops: {df['label'].nunique():,}")

for feature in required_features:
    takeaways.append(f"Missing {feature}: {int(df[feature].isna().sum()):,} rows ({df[feature].isna().mean() * 100:.2f}%)")

for item in takeaways:
    print("-", item)


- Total rows: 53,963
- Complete required-feature rows: 0
- Sources: 8
- Labels/crops: 159
- Missing area_size: 34,274 rows (63.51%)
- Missing soil_type: 30,963 rows (57.38%)
- Missing ph: 8,000 rows (14.82%)
- Missing n: 0 rows (0.00%)
- Missing p: 0 rows (0.00%)
- Missing k: 0 rows (0.00%)
- Missing soil_humidity: 32,096 rows (59.48%)
- Missing temp: 3,867 rows (7.17%)
- Missing rainfall: 11,867 rows (21.99%)
